# Proyecto 2: Análisis Exploratorio
## CommonLit - Evaluate Student Summaries


Reto seleccionado: **#12 - CommonLit: Evaluar resúmenes de estudiantes** (Procesamiento del Lenguaje Natural)

Competencia de Kaggle: [CommonLit - Evaluate Student Summaries](https://www.kaggle.com/competitions/commonlit-evaluate-student-summaries/data)

## 1. Situación problemática

Redactar un resumen es una de las habilidades más importantes que un estudiante desarrolla a lo largo de su formación académica. Resumir un texto obliga al estudiante a identificar la idea principal, distinguirla de los detalles secundarios y expresarla nuevamente con sus propias palabras de forma clara y coherente. Por esta razón, la habilidad de resumen está directamente relacionada con la comprensión lectora, y es especialmente relevante para estudiantes que están aprendiendo un segundo idioma o que presentan alguna dificultad de aprendizaje, ya que refuerza tanto la lectura crítica como la escritura.

A pesar de su importancia, los estudiantes de los grados 3 a 12 (educación primaria y secundaria en el sistema estadounidense) rara vez tienen suficientes oportunidades para practicar esta habilidad de forma constante. La razón principal es que evaluar un resumen es una tarea que consume mucho tiempo para los docentes: a diferencia de una pregunta de opción múltiple, calificar un resumen requiere que el profesor lea tanto el texto original como el resumen del estudiante, y luego juzgue de forma subjetiva qué tan bien se representaron las ideas principales y los detalles del texto fuente, además de la calidad del lenguaje empleado (claridad, precisión y fluidez). Cuando un docente tiene decenas o cientos de estudiantes, revisar resumen por resumen se vuelve una carga de trabajo poco sostenible, lo que reduce la frecuencia con la que se asignan este tipo de ejercicios y retrasa la retroalimentación que reciben los estudiantes.

En los últimos años han surgido técnicas de calificación automática de escritura (Automated Writing Evaluation), pero estas se han enfocado principalmente en géneros como el ensayo argumentativo o narrativo, donde el modelo solo necesita evaluar el texto del estudiante de forma aislada. La evaluación de resúmenes agrega una capa adicional de complejidad: el modelo no solo debe analizar la calidad interna del texto escrito por el estudiante, sino también compararlo contra un texto fuente más largo para determinar qué tan fielmente se representó su contenido. Las pocas técnicas existentes para evaluar resúmenes se han entrenado y probado sobre resúmenes generados automáticamente (por otros modelos de lenguaje), y no sobre escritura real de estudiantes, que presenta errores gramaticales, ideas incompletas, paráfrasis pobres y una gran variabilidad en extensión y estilo propia de la edad y el nivel de cada estudiante.

Esta situación plantea una oportunidad concreta: si se dispusiera de un modelo capaz de evaluar automáticamente, de forma confiable, tanto el contenido (*content*) como la redacción (*wording*) de un resumen escrito por un estudiante real, los docentes podrían asignar más ejercicios de resumen sin incrementar su carga de trabajo, y las plataformas educativas podrían ofrecer retroalimentación inmediata a los estudiantes mientras practican, acelerando su proceso de aprendizaje.

## 2. Problema científico

¿En qué medida es posible predecir de forma automática, mediante técnicas de Procesamiento del Lenguaje Natural (NLP) y aprendizaje automático supervisado, las puntuaciones de **contenido** (*content*) y **redacción** (*wording*) que un evaluador humano asignaría a un resumen escrito por un estudiante de los grados 3 a 12, utilizando únicamente el texto del resumen y el texto fuente (*prompt*) a partir del cual fue escrito?

De este problema general se desprenden preguntas más específicas que guiarán el análisis:

- ¿Qué características textuales del resumen (longitud, superposición léxica con el texto fuente, diversidad de vocabulario, estructura gramatical) se relacionan más fuertemente con la puntuación de *content* y cuáles con la de *wording*?
- ¿Las puntuaciones de *content* y *wording* están correlacionadas entre sí, o miden aspectos suficientemente distintos del resumen como para requerir un tratamiento independiente?
- ¿Un modelo entrenado con los resúmenes de un conjunto reducido de textos fuente (prompts) es capaz de generalizar su evaluación a resúmenes de textos fuente completamente distintos, dado que el conjunto de entrenamiento y el de prueba no comparten *prompts*?
- ¿Qué tanta variabilidad existe en la forma de escribir de estudiantes reales (longitud, errores, estilo) y cómo afecta esa variabilidad la dificultad de construir un modelo de evaluación confiable, en comparación con evaluar resúmenes generados por otros modelos de lenguaje?

Responder estas preguntas mediante un análisis exploratorio riguroso es el primer paso necesario antes de poder plantear y validar un modelo predictivo de evaluación automática de resúmenes.

## 3. Objetivos

### Objetivo general

Realizar un análisis exploratorio de los datos de la competencia *CommonLit: Evaluate Student Summaries*, con el fin de caracterizar los resúmenes escritos por estudiantes y los textos fuente de los que provienen, e identificar los patrones y relaciones entre las variables textuales y las puntuaciones de *content* y *wording* que sienten las bases para el desarrollo de un futuro modelo predictivo de evaluación automática.

### Objetivos específicos

1. Describir la estructura de los conjuntos de datos `summaries_train.csv` y `prompts_train.csv` (número de variables, número de observaciones y tipo de cada variable), y aplicar las tareas de limpieza y preprocesamiento necesarias (verificación de duplicados, valores faltantes y consistencia de los identificadores entre ambos conjuntos).
2. Calcular estadísticas descriptivas de las variables numéricas (`content`, `wording` y la longitud de los resúmenes en palabras/caracteres) y tablas de frecuencia para la variable categórica `prompt_id`, con el fin de cuantificar la distribución de las puntuaciones y de las observaciones por texto fuente.
3. Analizar, mediante gráficos exploratorios (histogramas, diagramas de caja y dispersión) y el cálculo de correlaciones, la relación entre la longitud/características del resumen y las puntuaciones de `content` y `wording`, así como la relación entre ambas puntuaciones entre sí y entre los distintos `prompt_id`, con el objetivo de identificar los factores que más explican la variabilidad en la calidad de los resúmenes.

## 4. Descripción de los datos

El conjunto de datos de la competencia está compuesto por cuatro archivos principales: `summaries_train.csv`, `prompts_train.csv`, `summaries_test.csv` y `prompts_test.csv`. Dado que los archivos de prueba (`test`) solo contienen 2 *prompts* y 4 resúmenes de ejemplo (son *placeholders* que Kaggle reemplaza por el conjunto de prueba real y oculto al momento de evaluar), el análisis exploratorio de este documento se centra en los archivos de entrenamiento, que son los que contienen datos reales y completos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

summaries_train = pd.read_csv("data/summaries_train.csv")
prompts_train = pd.read_csv("data/prompts_train.csv")

print("summaries_train:", summaries_train.shape)
print("prompts_train:", prompts_train.shape)

summaries_train: (7165, 5)
prompts_train: (4, 4)


`summaries_train.csv` tiene **7,165 observaciones y 5 variables**:

| Variable | Tipo | Descripción |
|---|---|---|
| `student_id` | Cualitativa nominal (identificador) | Identificador único del estudiante que escribió el resumen |
| `prompt_id` | Cualitativa nominal (categórica) | Identificador del texto fuente que el estudiante resumió; enlaza con `prompts_train.csv` |
| `text` | Cualitativa (texto libre) | Texto completo del resumen escrito por el estudiante |
| `content` | Cuantitativa continua (variable objetivo) | Puntuación de contenido del resumen (qué tan bien representa la idea principal y los detalles del texto fuente) |
| `wording` | Cuantitativa continua (variable objetivo) | Puntuación de redacción del resumen (claridad, precisión y fluidez del lenguaje) |

`prompts_train.csv` tiene **4 observaciones y 4 variables**:

| Variable | Tipo | Descripción |
|---|---|---|
| `prompt_id` | Cualitativa nominal (identificador) | Identificador único del texto fuente |
| `prompt_question` | Cualitativa (texto libre) | Pregunta específica que se le pidió responder al estudiante |
| `prompt_title` | Cualitativa nominal | Título corto del texto fuente |
| `prompt_text` | Cualitativa (texto libre) | Texto completo del material que los estudiantes debían resumir |

`content` y `wording` no son calificaciones en una escala fija (por ejemplo, de 1 a 5), sino puntuaciones continuas ya estandarizadas (pueden tomar valores negativos), obtenidas mediante un proceso de calificación comparativa entre resúmenes por parte de evaluadores expertos.

In [2]:
print("Tipos de dato - summaries_train:")
print(summaries_train.dtypes)
print("\nTipos de dato - prompts_train:")
print(prompts_train.dtypes)

Tipos de dato - summaries_train:
student_id     object
prompt_id      object
text           object
content       float64
wording       float64
dtype: object

Tipos de dato - prompts_train:
prompt_id          object
prompt_question    object
prompt_title       object
prompt_text        object
dtype: object


### 4.1 Limpieza y preprocesamiento

Se verifican valores faltantes, filas duplicadas y la consistencia de los identificadores `prompt_id` entre ambos archivos, y se generan variables derivadas de longitud de texto que se usarán en el análisis exploratorio.

In [3]:
# Valores faltantes
print("Valores faltantes - summaries_train:")
print(summaries_train.isna().sum())
print("\nValores faltantes - prompts_train:")
print(prompts_train.isna().sum())

# Duplicados
print("\nFilas duplicadas en summaries_train:", summaries_train.duplicated().sum())
print("student_id duplicados:", summaries_train["student_id"].duplicated().sum())

# Consistencia de prompt_id entre ambos datasets
prompts_en_summaries = set(summaries_train["prompt_id"].unique())
prompts_en_prompts = set(prompts_train["prompt_id"].unique())
print("\nprompt_id únicos en summaries_train:", prompts_en_summaries)
print("prompt_id únicos en prompts_train:", prompts_en_prompts)
print("¿Coinciden exactamente?", prompts_en_summaries == prompts_en_prompts)

Valores faltantes - summaries_train:
student_id    0
prompt_id     0
text          0
content       0
wording       0
dtype: int64

Valores faltantes - prompts_train:
prompt_id          0
prompt_question    0
prompt_title       0
prompt_text        0
dtype: int64

Filas duplicadas en summaries_train: 0
student_id duplicados: 0

prompt_id únicos en summaries_train: {'814d6b', 'ebad26', '39c16e', '3b9047'}
prompt_id únicos en prompts_train: {'814d6b', 'ebad26', '39c16e', '3b9047'}
¿Coinciden exactamente? True


In [4]:
# Variables derivadas de longitud de texto (necesarias para el análisis exploratorio)
summaries_train["text_len_words"] = summaries_train["text"].str.split().str.len()
summaries_train["text_len_chars"] = summaries_train["text"].str.len()

summaries_train[["text_len_words", "text_len_chars"]].describe()

,text_len_words,text_len_chars
count,7165.000000,7165.000000
mean,74.811724,418.776971
std,53.503992,307.833685
min,22.000000,114.000000
25%,39.000000,216.000000
50%,58.000000,320.000000
75%,92.000000,513.000000
max,647.000000,3940.000000


**Hallazgos de la limpieza de datos:**

- Ninguna de las 5 variables de `summaries_train` ni de las 4 variables de `prompts_train` presenta valores faltantes.
- No existen filas duplicadas ni `student_id` repetidos: cada una de las 7,165 observaciones corresponde a un resumen distinto de un estudiante distinto.
- Los 4 valores únicos de `prompt_id` en `summaries_train` coinciden exactamente con los 4 `prompt_id` de `prompts_train`, por lo que ambas tablas pueden combinarse (`merge`) sin pérdida ni generación de valores nulos.
- Los resúmenes tienen una longitud muy variable: entre 22 y 647 palabras (promedio ≈ 75 palabras, mediana ≈ 58), lo que confirma la heterogeneidad propia de la escritura real de estudiantes y sugiere que la longitud del resumen es una variable a considerar en el análisis exploratorio.

Dado que el conjunto de datos ya llega limpio (sin nulos ni duplicados), el preprocesamiento se limitó a verificar su integridad y a construir las variables derivadas de longitud de texto que se utilizarán en la siguiente sección.